# Taller de Cierre - Unidad I

**Asignatura:** Electiva IV - Gestion de Datos con Python

Este cuaderno resuelve las actividades 1 a 4 del taller. El archivo `Unidad1_Reto.json` debe estar en la misma carpeta de este cuaderno.

## 1. Cargar el archivo JSON

Se importa la libreria `json` y se carga la informacion de los estudiantes.

In [ ]:
import json
from pathlib import Path

ARCHIVO = Path('Unidad1_Reto.json')

with ARCHIVO.open(encoding='utf-8') as archivo:
    mydata = json.load(archivo)

print('Archivo cargado correctamente')
print('Tipo de dato:', type(mydata))


## Funciones de apoyo

Estas funciones permiten trabajar con los nombres y apellidos guardados como diccionarios en el JSON.

In [ ]:
def texto(valor):
    return '' if valor is None else str(valor).strip()


def valor_por_nombre(datos, palabra):
    for llave, valor in datos.items():
        if palabra in llave.casefold():
            return texto(valor)
    return ''


def estudiantes_de(datos):
    if isinstance(datos, list):
        return datos
    if 'estudiantes' in datos:
        return datos['estudiantes']
    return list(datos.values())


def partes_nombre(estudiante):
    nombres = estudiante.get('nombres', {})
    apellidos = estudiante.get('apellidos', {})
    primer_nombre = valor_por_nombre(nombres, 'primer')
    segundo_nombre = valor_por_nombre(nombres, 'segundo')
    primer_apellido = valor_por_nombre(apellidos, 'primer')
    segundo_apellido = valor_por_nombre(apellidos, 'segundo')
    return primer_nombre, segundo_nombre, primer_apellido, segundo_apellido


estudiantes = estudiantes_de(mydata)
print('Cantidad de estudiantes:', len(estudiantes))


## 2. Promedio de nota por asignatura

Se suman las notas de cada asignatura y se dividen por la cantidad de estudiantes que la cursaron.

In [ ]:
acumulados = {}

for estudiante in estudiantes:
    for asignatura in estudiante.get('asignaturas', []):
        nombre = asignatura['nombre']
        nota = float(asignatura['nota'])
        if nombre not in acumulados:
            acumulados[nombre] = {'suma': 0, 'cantidad': 0}
        acumulados[nombre]['suma'] += nota
        acumulados[nombre]['cantidad'] += 1

promedios_asignatura = {
    nombre: round(datos['suma'] / datos['cantidad'], 2)
    for nombre, datos in acumulados.items()
}

print('Promedio por asignatura:')
for nombre, promedio in promedios_asignatura.items():
    print(f'{nombre}: {promedio}')


## 3. Promedio por estudiante de asignaturas no retiradas

Solo se tienen en cuenta las asignaturas cuya llave `retirada` contiene el valor `No`. Al final se ordena la lista por primer apellido.

In [ ]:
promedios_estudiante = []

for estudiante in estudiantes:
    notas_cursadas = [
        float(asignatura['nota'])
        for asignatura in estudiante.get('asignaturas', [])
        if texto(asignatura.get('retirada')).casefold() == 'no'
    ]

    primer_nombre, _, primer_apellido, segundo_apellido = partes_nombre(estudiante)
    nombre_completo = ' '.join(
        parte for parte in (primer_nombre, primer_apellido, segundo_apellido) if parte
    )
    promedio = round(sum(notas_cursadas) / len(notas_cursadas), 2) if notas_cursadas else None

    promedios_estudiante.append({
        'codigo': estudiante.get('codigo'),
        'estudiante': nombre_completo,
        'promedio': promedio,
        '_apellido': primer_apellido.casefold()
    })

promedios_estudiante.sort(key=lambda estudiante: estudiante['_apellido'])
for estudiante in promedios_estudiante:
    del estudiante['_apellido']
    print(estudiante)


## 4. Estudiantes y correos institucionales

Con dos nombres se usa: inicial del primer nombre + inicial del segundo nombre + primer apellido + dos ultimos digitos del documento.

Con un nombre se usa: inicial del nombre + inicial del primer apellido + segundo apellido + dos ultimos digitos del documento.

In [ ]:
DOMINIO = '@uptc.edu.co'
lista_correos = []

for estudiante in estudiantes:
    primer_nombre, segundo_nombre, primer_apellido, segundo_apellido = partes_nombre(estudiante)
    documento = texto(estudiante.get('documento'))[-2:]

    if segundo_nombre:
        usuario = f'{primer_nombre[:1]}{segundo_nombre[:1]}.{primer_apellido}.{documento}'
    else:
        usuario = f'{primer_nombre[:1]}.{primer_apellido[:1]}.{segundo_apellido}.{documento}'

    nombre_completo = ' '.join(
        parte for parte in (primer_nombre, segundo_nombre, primer_apellido, segundo_apellido) if parte
    )
    lista_correos.append({
        'estudiante': nombre_completo,
        'correo': usuario.casefold() + DOMINIO
    })

for dato in lista_correos:
    print(dato)


## Actividad 5

Los ejercicios solicitados de Funciones, Cadenas, Listas, Conjuntos y Diccionarios estan desarrollados en los cuadernos de repaso de la Unidad I.